In [1]:
from pyspark.sql import functions as F

StatementMeta(, 7b9694c3-c301-48f0-910c-31e2eb82a649, 3, Finished, Available, Finished, False)

In [2]:
# ============================================================
# NB03_Lynch_Silver_Features (DEFENSIVE / BUG-FREE)
# Builds one current feature row per in-scope company using
# Yahoo data previously ingested by NB02.
#
# Output: silver_lynch_features
# ============================================================

import re
import numpy as np
import pandas as pd
from datetime import datetime, date
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType,
    IntegerType, BooleanType
)

RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
RUN_ID = str(uuid.uuid4())

# ------------------------------------------------------------
# 1. Load configuration with defaults
# ------------------------------------------------------------
try:
    params = (
        spark.read.format("csv")
        .option("header", "true")
        .load("Files/config/lynch_parameters.csv")
        .toPandas()
        .set_index("parameter")["value"]
        .to_dict()
    )
except Exception as e:
    print(f"Could not read config; using defaults. Details: {e}")
    params = {}

def param_float(name, default):
    try:
        return float(params.get(name, default))
    except (TypeError, ValueError):
        return float(default)

MIN_MCAP_CR = param_float("min_market_cap_cr", 2000)
SLOW_MIN    = param_float("slow_eps_cagr_min", 0.03)
SLOW_MAX    = param_float("slow_eps_cagr_max", 0.12)
STAL_MIN    = param_float("stalwart_eps_cagr_min", 0.08)
STAL_MAX    = param_float("stalwart_eps_cagr_max", 0.16)

# ------------------------------------------------------------
# 2. Load Yahoo Bronze tables and in-scope universe
# ------------------------------------------------------------
universe = (
    spark.table("bronze_universe")
    .filter("in_scope = true")
    .select("symbol", "yahoo_symbol", "company_name", "industry", "rule_set")
    .dropDuplicates(["symbol"])
    .toPandas()
)

if universe.empty:
    raise RuntimeError("bronze_universe has no in-scope rows. Run NB01 first.")

fund = spark.table("bronze_yahoo_fundamentals_long").toPandas()
prices = (
    spark.table("bronze_yahoo_prices")
    .select("symbol", "trade_date", "close")
    .toPandas()
)
dividends = spark.table("bronze_yahoo_dividends").toPandas()
market = spark.table("bronze_yahoo_market_snapshot").toPandas()

if fund.empty:
    raise RuntimeError("bronze_yahoo_fundamentals_long is empty. Run NB02 first.")

# Normalize basic types
fund["period_type"] = fund["period_type"].astype(str).str.upper()
fund["statement_type"] = fund["statement_type"].astype(str).str.upper()
fund["period_end"] = pd.to_datetime(fund["period_end"], errors="coerce")
fund["value"] = pd.to_numeric(fund["value"], errors="coerce")
fund = fund.dropna(subset=["symbol", "period_end", "value"])

prices["trade_date"] = pd.to_datetime(prices["trade_date"], errors="coerce")
prices["close"] = pd.to_numeric(prices["close"], errors="coerce")
prices = prices.dropna(subset=["symbol", "trade_date", "close"])
prices = prices[prices["close"] > 0]
prices = prices.drop_duplicates(["symbol", "trade_date"]).sort_values(["symbol", "trade_date"])

dividends["ex_date"] = pd.to_datetime(dividends["ex_date"], errors="coerce")
dividends["dividend"] = pd.to_numeric(dividends["dividend"], errors="coerce")
dividends = dividends.dropna(subset=["symbol", "ex_date", "dividend"])
dividends = dividends[dividends["dividend"] > 0]
dividends = dividends.drop_duplicates(["symbol", "ex_date"])

market = market.drop_duplicates(["symbol"], keep="last")

print(f"In-scope companies        : {len(universe)}")
print(f"Fundamental source rows   : {len(fund):,}")
print(f"Price rows                : {len(prices):,}")
print(f"Dividend rows             : {len(dividends):,}")

# ------------------------------------------------------------
# 3. Map Yahoo source fields to canonical metric names
# ------------------------------------------------------------
def normalize_label(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def canonical_metric(source_field, statement_type):
    key = normalize_label(source_field)
    st = str(statement_type).upper()

    if "INCOME" in st:
        if key in {"dilutedeps", "dilutedepscontinuingoperations", "dilutedepscontinuinganddiscontinuedoperations"}:
            return "eps_diluted"
        if key in {"basiceps", "basicepscontinuingoperations"}:
            return "eps_basic"
        if key in {"totalrevenue", "operatingrevenue", "totalrevenuefromoperations", "revenue"}:
            return "revenue"
        if key in {"ebit", "operatingincome"}:
            return "ebit"
        if key in {"netincome", "netincomecommonstockholders", "netincomecommonstockholdersincludingextraordinaryitems", "netincomecontinuingoperations"}:
            return "net_income"
        if "interestexpense" in key:
            return "interest_expense"

    if "BALANCE" in st:
        if key in {"stockholdersequity", "commonstockequity", "totalequitygrossminorityinterest", "totalequity"}:
            return "equity"
        if key == "totaldebt":
            return "total_debt"
        if key in {"longtermdebt", "longtermdebtandcapitalleaseobligation"}:
            return "long_debt"
        if key in {"currentdebt", "shorttermdebt", "currentdebtandcapitalleaseobligation"}:
            return "short_debt"
        if key in {"cashandcashequivalents", "cashcashequivalentsandshortterminvestments", "cashandshortterminvestments"}:
            return "cash"
        if key == "inventory":
            return "inventory"
        if key in {"ordinarysharesnumber", "shareissued"}:
            return "shares"

    if "CASH" in st or "FLOW" in st:
        if key in {"operatingcashflow", "cashflowfromcontinuingoperatingactivities", "cashfromoperatingactivities"}:
            return "ocf"
        if "capitalexpenditure" in key:
            return "capex"
        if key == "freecashflow":
            return "fcf"
        if "cashdividendspaid" in key or key == "dividendspaid":
            return "dividends_paid"
        if "depreciationandamortization" in key or key == "depreciation":
            return "depreciation"

    return None

def source_priority(source_field, metric):
    key = normalize_label(source_field)
    if metric == "eps":
        return 0 if "diluted" in key else 1
    if metric == "total_debt":
        return 0 if key == "totaldebt" else 1
    if metric == "equity":
        if key == "stockholdersequity": return 0
        if key == "commonstockequity": return 1
        return 2
    return 0

fund["metric"] = fund.apply(lambda r: canonical_metric(r["source_field"], r["statement_type"]), axis=1)
fund = fund.dropna(subset=["metric"]).copy()
fund.loc[fund["metric"].isin(["eps_diluted", "eps_basic"]), "metric"] = "eps"
fund["priority"] = fund.apply(lambda r: source_priority(r["source_field"], r["metric"]), axis=1)

fund = fund.sort_values(["symbol", "period_type", "period_end", "metric", "priority"]).drop_duplicates(
    ["symbol", "period_type", "period_end", "metric"], keep="first"
)

# ------------------------------------------------------------
# 4. Create wide annual and quarterly statements
# ------------------------------------------------------------
def pivot_statement(data):
    if data.empty:
        return pd.DataFrame(columns=["symbol", "period_end"])
    out = data.pivot_table(
        index=["symbol", "period_end"],
        columns="metric",
        values="value",
        aggfunc="first"
    ).reset_index()
    out.columns = [c if isinstance(c, str) else str(c) for c in out.columns]
    return out

annual = pivot_statement(fund[fund["period_type"] == "ANNUAL"])
quarterly = pivot_statement(fund[fund["period_type"] == "QUARTERLY"])
balance_long = fund[fund["statement_type"].str.contains("BALANCE", na=False)].copy()
balance = pivot_statement(balance_long)

def make_symbol_dict(df):
    if df.empty or "symbol" not in df.columns or "period_end" not in df.columns:
        return {}
    return {str(s): g.sort_values("period_end").reset_index(drop=True) for s, g in df.groupby("symbol")}

annual_by_symbol = make_symbol_dict(annual)
quarterly_by_symbol = make_symbol_dict(quarterly)
balance_by_symbol = make_symbol_dict(balance)

# ------------------------------------------------------------
# 5. Price and dividend lookup helpers
# ------------------------------------------------------------
prices_by_symbol = {str(s): g.sort_values("trade_date").reset_index(drop=True) for s, g in prices.groupby("symbol")}
dividends_by_symbol = {str(s): g.sort_values("ex_date").reset_index(drop=True) for s, g in dividends.groupby("symbol")}

def numeric_series(frame, column):
    if frame is None or frame.empty or column not in frame.columns:
        return pd.Series(dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce").dropna()

def safe_float(value, default=None):
    try:
        f = float(value)
        if not np.isfinite(f):
            return default
        return f
    except (TypeError, ValueError):
        return default

def cagr_from_frame(frame, column, periods=4):
    if frame is None or frame.empty or column not in frame.columns or "period_end" not in frame.columns:
        return None
    x = frame[["period_end", column]].copy()
    x[column] = pd.to_numeric(x[column], errors="coerce")
    x = x.dropna().sort_values("period_end").tail(periods)
    if len(x) < periods:
        return None
    first = float(x.iloc[0][column])
    last = float(x.iloc[-1][column])
    elapsed_years = (pd.to_datetime(x.iloc[-1]["period_end"]) - pd.to_datetime(x.iloc[0]["period_end"])).days / 365.25
    if first <= 0 or last <= 0 or elapsed_years <= 0:
        return None
    return (last / first) ** (1 / elapsed_years) - 1

def close_on_or_before(symbol, target_date, max_days=45):
    g = prices_by_symbol.get(str(symbol))
    if g is None or g.empty:
        return None
    target = pd.Timestamp(target_date)
    prior = g[g["trade_date"] <= target]
    if prior.empty:
        return None
    row = prior.iloc[-1]
    if (target - row["trade_date"]).days > max_days:
        return None
    return safe_float(row["close"])

# ------------------------------------------------------------
# 6. Dividend features
# ------------------------------------------------------------
def dividend_features(symbol, latest_price_date, current_price):
    result = {
        "ttm_dividend_per_share": None,
        "ttm_dividend_yield": None,
        "dividend_years_10": 0,
        "dividend_history_years": 0,
        "dividend_no_cut_last5": None,
        "dps_cagr_5y": None,
        "median_dividend_yield_10y": None,
        "yield_history_years": 0,
    }

    dg = dividends_by_symbol.get(str(symbol))
    if latest_price_date is None or current_price is None or current_price <= 0:
        return result

    latest_date = pd.Timestamp(latest_price_date)

    if dg is None or dg.empty:
        result["ttm_dividend_per_share"] = 0.0
        result["ttm_dividend_yield"] = 0.0
        dividend_events = pd.DataFrame(columns=["ex_date", "dividend"])
    else:
        dividend_events = dg[["ex_date", "dividend"]].copy()
        dividend_events["ex_date"] = pd.to_datetime(dividend_events["ex_date"])
        dividend_events["dividend"] = pd.to_numeric(dividend_events["dividend"], errors="coerce")
        dividend_events = dividend_events.dropna()

        ttm_start = latest_date - pd.Timedelta(days=365)
        ttm_dps = float(
            dividend_events.loc[
                (dividend_events["ex_date"] > ttm_start) & (dividend_events["ex_date"] <= latest_date),
                "dividend"
            ].sum()
        )
        result["ttm_dividend_per_share"] = ttm_dps
        result["ttm_dividend_yield"] = ttm_dps / current_price

    latest_fy_end_year = latest_date.year if latest_date.month >= 4 else latest_date.year - 1
    fy_years = list(range(latest_fy_end_year - 9, latest_fy_end_year + 1))

    dps_by_fy = {y: 0.0 for y in fy_years}
    if not dividend_events.empty:
        for _, d in dividend_events.iterrows():
            ex_date = pd.Timestamp(d["ex_date"])
            fy_end_year = ex_date.year + 1 if ex_date.month >= 4 else ex_date.year
            if fy_end_year in dps_by_fy:
                dps_by_fy[fy_end_year] += float(d["dividend"])

    annual_yields = []
    observable_years = []

    for y in fy_years:
        fy_end = pd.Timestamp(year=y, month=3, day=31)
        fy_price = close_on_or_before(symbol, fy_end, max_days=60)
        if fy_price is not None and fy_price > 0:
            observable_years.append(y)
            annual_yields.append(dps_by_fy[y] / fy_price)

    result["yield_history_years"] = len(observable_years)
    result["dividend_history_years"] = len(observable_years)
    result["dividend_years_10"] = int(sum(1 for y in observable_years if dps_by_fy[y] > 0))

    if annual_yields:
        result["median_dividend_yield_10y"] = float(np.median(annual_yields))

    if len(observable_years) >= 5:
        recent_years = observable_years[-5:]
        recent_dps = [dps_by_fy[y] for y in recent_years]
        result["dividend_no_cut_last5"] = (
            all(x > 0 for x in recent_dps)
            and all(recent_dps[i] >= recent_dps[i - 1] for i in range(1, len(recent_dps)))
        )

    if len(observable_years) >= 6:
        first_dps = dps_by_fy[observable_years[-6]]
        last_dps = dps_by_fy[observable_years[-1]]
        if first_dps > 0 and last_dps > 0:
            result["dps_cagr_5y"] = (last_dps / first_dps) ** (1 / 5) - 1

    return result

# ------------------------------------------------------------
# 7. Build one Silver feature row per in-scope symbol
# ------------------------------------------------------------
market_by_symbol = {str(r["symbol"]): r for _, r in market.drop_duplicates("symbol", keep="last").iterrows()}
feature_rows = []

for _, u in universe.iterrows():
    symbol = str(u["symbol"])
    notes = []

    ag = annual_by_symbol.get(symbol, pd.DataFrame(columns=["symbol", "period_end"]))
    qg = quarterly_by_symbol.get(symbol, pd.DataFrame(columns=["symbol", "period_end"]))
    bg = balance_by_symbol.get(symbol, pd.DataFrame(columns=["symbol", "period_end"]))
    pg = prices_by_symbol.get(symbol, pd.DataFrame(columns=["symbol", "trade_date", "close"]))

    # Latest price and date
    if pg is not None and not pg.empty:
        latest_price_row = pg.iloc[-1]
        price_date = pd.to_datetime(latest_price_row["trade_date"]).date()
        current_price = safe_float(latest_price_row["close"])
    else:
        price_date, current_price = None, None
        notes.append("No price history")

    # Market Cap
    mr = market_by_symbol.get(symbol, {})
    market_cap = safe_float(mr.get("market_cap"))
    market_cap_cr = market_cap / 1e7 if market_cap is not None else None

    # Annual history
    annual_periods = len(ag)
    eps_cagr_3y = cagr_from_frame(ag, "eps", periods=4)
    revenue_cagr_3y = cagr_from_frame(ag, "revenue", periods=4)
    inventory_cagr_3y = cagr_from_frame(ag, "inventory", periods=4)

    recent4 = ag.tail(4).copy() if not ag.empty else pd.DataFrame()
    eps_s = numeric_series(recent4, "eps")
    ni_s = numeric_series(recent4, "net_income")
    positive_earnings_years_4 = int((ni_s > 0).sum()) if len(ni_s) else None
    eps_periods_available = len(eps_s)

    # ROE
    min_roe_4y = None
    roe_years_available = 0
    if not recent4.empty and {"net_income", "equity"}.issubset(recent4.columns):
        roe_data = recent4[["net_income", "equity"]].dropna()
        roe_data = roe_data[roe_data["equity"] > 0]
        if len(roe_data):
            roes = roe_data["net_income"] / roe_data["equity"]
            min_roe_4y = float(roes.min())
            roe_years_available = int(len(roes))

    # Balance Sheet
    balance_source = bg if not bg.empty else ag
    latest_balance = (
        balance_source.sort_values("period_end").iloc[-1]
        if not balance_source.empty and "period_end" in balance_source.columns
        else pd.Series(dtype="float64")
    )

    total_equity = safe_float(latest_balance.get("equity"))
    total_debt = safe_float(latest_balance.get("total_debt"))
    if total_debt is None:
        long_debt = safe_float(latest_balance.get("long_debt"), 0.0)
        short_debt = safe_float(latest_balance.get("short_debt"), 0.0)
        total_debt = long_debt + short_debt

    debt_to_equity = (
        total_debt / total_equity
        if total_debt is not None and total_equity is not None and total_equity > 0
        else None
    )

    # Annual EBIT / Interest
    latest_annual = (
        ag.sort_values("period_end").iloc[-1]
        if not ag.empty and "period_end" in ag.columns
        else pd.Series(dtype="float64")
    )
    ebit = safe_float(latest_annual.get("ebit"))
    interest = safe_float(latest_annual.get("interest_expense"))
    interest_coverage = (
        ebit / abs(interest)
        if ebit is not None and interest is not None and abs(interest) > 0
        else None
    )

    # OCF / Net Income
    ocf_to_ni_4y = None
    ocf_ni_years = 0
    if not recent4.empty and {"ocf", "net_income"}.issubset(recent4.columns):
        matched = recent4[["ocf", "net_income"]].dropna()
        matched = matched[matched["net_income"] > 0]
        ocf_ni_years = int(len(matched))
        if len(matched) >= 3 and matched["net_income"].sum() > 0:
            ocf_to_ni_4y = float(matched["ocf"].sum() / matched["net_income"].sum())

    # FCF payout & Capex / Depreciation
    fcf_payouts, capex_dep = [], []
    if not recent4.empty:
        for _, yr in recent4.iterrows():
            ocf = safe_float(yr.get("ocf"))
            fcf = safe_float(yr.get("fcf"))
            capex = safe_float(yr.get("capex"))
            dep = safe_float(yr.get("depreciation"))
            dividends_paid = safe_float(yr.get("dividends_paid"))

            if fcf is None and ocf is not None and capex is not None:
                fcf = ocf - abs(capex)
            if fcf is not None and fcf > 0 and dividends_paid is not None:
                fcf_payouts.append(abs(dividends_paid) / fcf)
            if capex is not None and dep is not None and abs(dep) > 0:
                capex_dep.append(abs(capex) / abs(dep))

    fcf_payout_3y = float(np.mean(fcf_payouts[-3:])) if len(fcf_payouts) >= 2 else None
    capex_to_depreciation_4y = float(np.mean(capex_dep[-4:])) if len(capex_dep) >= 2 else None

    # Margins
    margin_declines_3y = None
    if not ag.empty and {"ebit", "revenue", "period_end"}.issubset(ag.columns):
        margins = ag[["period_end", "ebit", "revenue"]].dropna().sort_values("period_end").tail(4)
        margins = margins[margins["revenue"] > 0]
        if len(margins) >= 2:
            margin_values = (margins["ebit"] / margins["revenue"]).tolist()
            margin_declines_3y = int(sum(1 for i in range(1, len(margin_values)) if margin_values[i] < margin_values[i - 1]))

    # Dividends
    divf = dividend_features(symbol, price_date, current_price)

    # --------------------------------------------------------
    # P/E CALCULATION (CRITICAL DEFENSIVE FIX)
    # --------------------------------------------------------
    pe_current, pe_basis = None, "UNAVAILABLE"
    ttm_eps_valid = False

    # Only attempt quarterly TTM if qg has both period_end and eps columns
    if not qg.empty and {"period_end", "eps"}.issubset(qg.columns):
        q_sorted = qg.dropna(subset=["eps"]).sort_values("period_end")
        q_eps = numeric_series(q_sorted.tail(4), "eps")
        if len(q_eps) == 4:
            q_dates = q_sorted.tail(4)["period_end"]
            if len(q_dates) == 4:
                span_days = (pd.to_datetime(q_dates.iloc[-1]) - pd.to_datetime(q_dates.iloc[0])).days
                if 250 <= span_days <= 430:
                    ttm_eps = float(q_eps.sum())
                    if ttm_eps > 0 and current_price is not None:
                        pe_current = current_price / ttm_eps
                        pe_basis = "QUARTERLY_TTM"
                        ttm_eps_valid = True

    # Fallback to latest annual EPS
    if not ttm_eps_valid:
        latest_annual_eps = safe_float(latest_annual.get("eps"))
        if latest_annual_eps is not None and latest_annual_eps > 0 and current_price:
            pe_current = current_price / latest_annual_eps
            pe_basis = "LATEST_ANNUAL_EPS"

    # Historical P/E
    historical_pes = []
    if not ag.empty and {"eps", "period_end"}.issubset(ag.columns):
        for _, yr in ag[["period_end", "eps"]].dropna().iterrows():
            yr_eps = safe_float(yr["eps"])
            if yr_eps is None or yr_eps <= 0:
                continue
            yr_price = close_on_or_before(symbol, yr["period_end"], max_days=60)
            if yr_price is not None:
                historical_pes.append(yr_price / yr_eps)

    pe_own_median_4y = float(np.median(historical_pes)) if len(historical_pes) >= 3 else None
    pe_vs_own_median = (
        pe_current / pe_own_median_4y
        if pe_current is not None and pe_own_median_4y and pe_own_median_4y > 0
        else None
    )

    # Dividend-Adjusted PEG (Lynch): Higher is better!
    # Formula = (EPS Growth % + Dividend Yield %) / P/E
    growth_plus_yield_pct = None
    if eps_cagr_3y is not None and divf["ttm_dividend_yield"] is not None:
        growth_plus_yield_pct = (eps_cagr_3y * 100) + (divf["ttm_dividend_yield"] * 100)

    dividend_adjusted_peg = (
        growth_plus_yield_pct / pe_current
        if pe_current is not None and pe_current > 0 and growth_plus_yield_pct is not None and growth_plus_yield_pct > 0
        else None
    )

    # Data coverage notes
    if annual_periods < 4:
        notes.append(f"Only {annual_periods} annual statement periods")
    if eps_periods_available < 4:
        notes.append("Insufficient annual EPS history for 3-year CAGR")
    if roe_years_available < 4:
        notes.append(f"ROE available for {roe_years_available} of last 4 periods")
    if divf["yield_history_years"] < 8:
        notes.append(f"Own-yield history available for {divf['yield_history_years']} years")
    if pe_basis == "LATEST_ANNUAL_EPS":
        notes.append("P/E uses latest annual EPS, not TTM EPS")
    if pe_basis == "UNAVAILABLE":
        notes.append("P/E unavailable")

    # Classifications
    category = "OUT_OF_SCOPE"
    if eps_cagr_3y is None or eps_periods_available < 4:
        category = "INSUFFICIENT_HISTORY"
    elif SLOW_MIN <= eps_cagr_3y < SLOW_MAX:
        category = "PROVISIONAL_SLOW_GROWER"
    elif STAL_MIN <= eps_cagr_3y <= STAL_MAX:
        category = "PROVISIONAL_STALWART"
    elif eps_cagr_3y > STAL_MAX:
        category = "FAST_GROWER_OUT_OF_SCOPE"
    else:
        category = "LOW_GROWTH_OUT_OF_SCOPE"

    if price_date is None or current_price is None:
        data_status = "INSUFFICIENT_PRICE_DATA"
    elif annual_periods < 4:
        data_status = "LIMITED_FUNDAMENTAL_HISTORY"
    else:
        data_status = "FEATURES_CALCULATED"

    feature_rows.append({
        "symbol": symbol,
        "company_name": u["company_name"],
        "industry": u["industry"],
        "rule_set": u["rule_set"],
        "price_date": price_date,
        "current_price": current_price,
        "market_cap_cr": market_cap_cr,
        "annual_periods_available": annual_periods,
        "eps_periods_available": eps_periods_available,
        "eps_cagr_3y": eps_cagr_3y,
        "positive_earnings_years_4": positive_earnings_years_4,
        "min_roe_4y": min_roe_4y,
        "roe_years_available": roe_years_available,
        "debt_to_equity": debt_to_equity,
        "interest_coverage": interest_coverage,
        "ocf_to_ni_4y": ocf_to_ni_4y,
        "ocf_ni_years": ocf_ni_years,
        "fcf_payout_3y": fcf_payout_3y,
        "capex_to_depreciation_4y": capex_to_depreciation_4y,
        "revenue_cagr_3y": revenue_cagr_3y,
        "inventory_cagr_3y": inventory_cagr_3y,
        "inventory_sales_growth_gap": (
            inventory_cagr_3y - revenue_cagr_3y
            if inventory_cagr_3y is not None and revenue_cagr_3y is not None
            else None
        ),
        "margin_declines_3y": margin_declines_3y,
        "ttm_dividend_per_share": divf["ttm_dividend_per_share"],
        "ttm_dividend_yield": divf["ttm_dividend_yield"],
        "dividend_years_10": divf["dividend_years_10"],
        "dividend_history_years": divf["dividend_history_years"],
        "dividend_no_cut_last5": divf["dividend_no_cut_last5"],
        "dps_cagr_5y": divf["dps_cagr_5y"],
        "median_dividend_yield_10y": divf["median_dividend_yield_10y"],
        "yield_history_years": divf["yield_history_years"],
        "pe_current": pe_current,
        "pe_basis": pe_basis,
        "pe_own_median_4y": pe_own_median_4y,
        "pe_history_points": len(historical_pes),
        "pe_vs_own_median": pe_vs_own_median,
        "dividend_adjusted_peg": dividend_adjusted_peg,
        "category_provisional": category,
        "data_status": data_status,
        "data_notes": "; ".join(notes),
        "screened_at": RUN_TS,
        "run_id": RUN_ID,
    })

# ------------------------------------------------------------
# 8. Industry peer P/E comparison
# ------------------------------------------------------------
result = pd.DataFrame(feature_rows)

if not result.empty:
    result["industry_peer_median_pe"] = result.groupby("industry")["pe_current"].transform("median")
    result["pe_vs_peer_median"] = np.where(
        result["industry_peer_median_pe"] > 0,
        result["pe_current"] / result["industry_peer_median_pe"],
        np.nan
    )
else:
    result["industry_peer_median_pe"] = pd.Series(dtype="float64")
    result["pe_vs_peer_median"] = pd.Series(dtype="float64")

# ------------------------------------------------------------
# 9. Save to Delta Table
# ------------------------------------------------------------
OUTPUT_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("company_name", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("rule_set", StringType(), True),
    StructField("price_date", DateType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap_cr", DoubleType(), True),
    StructField("annual_periods_available", IntegerType(), True),
    StructField("eps_periods_available", IntegerType(), True),
    StructField("eps_cagr_3y", DoubleType(), True),
    StructField("positive_earnings_years_4", IntegerType(), True),
    StructField("min_roe_4y", DoubleType(), True),
    StructField("roe_years_available", IntegerType(), True),
    StructField("debt_to_equity", DoubleType(), True),
    StructField("interest_coverage", DoubleType(), True),
    StructField("ocf_to_ni_4y", DoubleType(), True),
    StructField("ocf_ni_years", IntegerType(), True),
    StructField("fcf_payout_3y", DoubleType(), True),
    StructField("capex_to_depreciation_4y", DoubleType(), True),
    StructField("revenue_cagr_3y", DoubleType(), True),
    StructField("inventory_cagr_3y", DoubleType(), True),
    StructField("inventory_sales_growth_gap", DoubleType(), True),
    StructField("margin_declines_3y", IntegerType(), True),
    StructField("ttm_dividend_per_share", DoubleType(), True),
    StructField("ttm_dividend_yield", DoubleType(), True),
    StructField("dividend_years_10", IntegerType(), True),
    StructField("dividend_history_years", IntegerType(), True),
    StructField("dividend_no_cut_last5", BooleanType(), True),
    StructField("dps_cagr_5y", DoubleType(), True),
    StructField("median_dividend_yield_10y", DoubleType(), True),
    StructField("yield_history_years", IntegerType(), True),
    StructField("pe_current", DoubleType(), True),
    StructField("pe_basis", StringType(), True),
    StructField("pe_own_median_4y", DoubleType(), True),
    StructField("pe_history_points", IntegerType(), True),
    StructField("pe_vs_own_median", DoubleType(), True),
    StructField("industry_peer_median_pe", DoubleType(), True),
    StructField("pe_vs_peer_median", DoubleType(), True),
    StructField("dividend_adjusted_peg", DoubleType(), True),
    StructField("category_provisional", StringType(), True),
    StructField("data_status", StringType(), True),
    StructField("data_notes", StringType(), True),
    StructField("screened_at", StringType(), True),
    StructField("run_id", StringType(), True),
])

FIELD_TYPES = {f.name: f.dataType for f in OUTPUT_SCHEMA.fields}

def clean_val(f_name, v):
    """Convert a pandas value to the exact Python type the Spark schema expects."""
    dtype = FIELD_TYPES[f_name]

    if v is None:
        return None
    try:
        if pd.isna(v):          # NaN, NaT, pd.NA
            return None
    except (TypeError, ValueError):
        pass

    try:
        if isinstance(dtype, IntegerType):
            return int(v)
        if isinstance(dtype, DoubleType):
            f = float(v)
            return f if np.isfinite(f) else None
        if isinstance(dtype, BooleanType):
            return bool(v)
        if isinstance(dtype, DateType):
            if isinstance(v, datetime):      # includes pd.Timestamp
                return v.date()
            if isinstance(v, date):
                return v
            return pd.to_datetime(v).date()
        if isinstance(dtype, StringType):
            return str(v)
    except (TypeError, ValueError):
        return None

    return v

cols = [f.name for f in OUTPUT_SCHEMA.fields]
records = [tuple(clean_val(c, r.get(c)) for c in cols) for _, r in result.iterrows()]

silver_df = spark.createDataFrame(records, schema=OUTPUT_SCHEMA)

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_lynch_features")
)

print(f"\nSUCCESS: Saved silver_lynch_features with {len(records)} rows!")

print("\nProvisional category counts:")
display(
    silver_df.groupBy("category_provisional")
    .count()
    .orderBy(F.desc("count"))
)

print("\nData status counts:")
display(
    silver_df.groupBy("data_status")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, 7b9694c3-c301-48f0-910c-31e2eb82a649, 4, Finished, Available, Finished, False)

/tmp/ipykernel_6219/2263513578.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


In-scope companies        : 343
Fundamental source rows   : 371,941
Price rows                : 1,348,119
Dividend rows             : 7,178

SUCCESS: Saved silver_lynch_features with 343 rows!

Provisional category counts:


SynapseWidget(Synapse.DataFrame, b698bbe0-ae37-46f8-aeb0-08ded9c8ac8d)


Data status counts:


SynapseWidget(Synapse.DataFrame, 9cd4353a-267e-4064-94c1-d08767ec8c63)

In [3]:
%%sql
SELECT
    category_provisional,
    data_status,
    COUNT(*) AS companies
FROM silver_lynch_features
GROUP BY category_provisional, data_status
ORDER BY category_provisional, data_status;

StatementMeta(, 7b9694c3-c301-48f0-910c-31e2eb82a649, 5, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 3 fields>